# OTDR Model Testing with Best Checkpoint
This notebook loads the best checkpoint and evaluates the model on test data


In [2]:
import os
import sys
sys.path.append(".")

# Set GPU device
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
import numpy as np
import mmcv
from mmcv.utils import Config
from mmcv.runner import load_checkpoint
from tqdm import tqdm

# Import custom modules
import backbone
import dataset
import head
import segmentor
from mmseg.datasets import build_dataset, build_dataloader
from mmseg.models import build_segmentor
from scripts.metrics import create_evaluator


## 1. Configuration and Setup


In [3]:
# Configuration paths
config_path = '/home/mjl/wangshuo/otdr/configs/convmae/custom_train.py'  # Update this to your config file
checkpoint_path = '/home/mjl/wangshuo/otdr/out/otdr_full_bs16/checkpoints/best.pth'  # Update this to your best checkpoint

# Load config
cfg = Config.fromfile(config_path)

# Test configuration
threshold = 0.5  # Binary threshold
device = 'cuda:0'

print(f"Config loaded from: {config_path}")
print(f"Checkpoint path: {checkpoint_path}")
print(f"Device: {device}")
print(f"Threshold: {threshold}")


Config loaded from: /home/mjl/wangshuo/otdr/configs/convmae/custom_train.py
Checkpoint path: /home/mjl/wangshuo/otdr/out/otdr_full_bs16/checkpoints/best.pth
Device: cuda:0
Threshold: 0.5


## 2. Build Model and Load Checkpoint


In [4]:
# Build model
cfg.model.train_cfg = None
model = build_segmentor(cfg.model, test_cfg=cfg.get('test_cfg'))
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# 计算可训练参数量
total_params = count_parameters(model)
print(f"Total trainable parameters: {total_params:,}")
print(f"Total trainable parameters: {total_params/1e6:.2f}M")
# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location='cpu')
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

# Load state dict
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Missing keys: {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")

# Move to device
model = model.to(device)
model.eval()

print("\nModel loaded successfully!")
print(f"Model type: {type(model).__name__}")


Total trainable parameters: 104,485,156
Total trainable parameters: 104.49M
Missing keys: 0
Unexpected keys: 0

Model loaded successfully!
Model type: EncoderDecoderVideo


## 3. Build Test Dataset


In [7]:
# Build test dataset
test_dataset = build_dataset(cfg.data.test, dict(test_mode=True))

# Build dataloader
test_loader = build_dataloader(
    test_dataset,
    samples_per_gpu=1,
    workers_per_gpu=cfg.data.workers_per_gpu,
    dist=False,
    shuffle=False
)

print(f"Test dataset size: {len(test_dataset)}")
print(f"Number of batches: {len(test_loader)}")
print(f"Number of classes: {cfg.model.decode_head.num_classes}")


2025-10-20 11:58:43,771 - mmseg - INFO - Loaded 4292 images


Test dataset size: 4292
Number of batches: 4292
Number of classes: 2


## 4. Run Inference and Evaluation


In [11]:
# Create evaluator
num_classes = cfg.model.decode_head.num_classes
ignore_index = cfg.get('ignore_index', 255)
evaluator = create_evaluator(
    num_classes=num_classes,
    ignore_index=ignore_index,
    metrics=['mIoU']
)

# Also collect confusion matrix for binary classification
conf_mat = np.zeros((num_classes, num_classes), dtype=np.int64)

print("Starting inference...")
print(f"Number of classes: {num_classes}")
print(f"Ignore index: {ignore_index}")

model.eval()
# Run inference
for i, data_batch in enumerate(tqdm(test_loader, desc="Testing")):
    # Get images and metadata
    imgs = [img.to(device) for img in data_batch['img']]
    img_metas = [meta.data[0] for meta in data_batch['img_metas']]
    
    # Run inference
    with torch.no_grad():
        seg_pred, _, _ = model.forward_test(imgs, img_metas)
        if seg_pred is not None:
            seg_pred = seg_pred[0]  # Get first batch element
    
    # Load ground truth
    batch_gts = []
    for img_meta in img_metas[0]:
        gt_path = os.path.join(test_dataset.ann_dir, img_meta['ann']['seg_map'])
        gt_seg = mmcv.imread(gt_path, flag='unchanged', backend='pillow')
        if gt_seg is None:
            print(f"Warning: GT not found at {gt_path}")
            continue
        batch_gts.append(gt_seg)
    
    if len(batch_gts) == 0:
        continue
        
    batch_gts = np.stack(batch_gts, axis=0)
    
    # Add to evaluator
    evaluator.add_batch(seg_pred, batch_gts)
    
    # Update confusion matrix for binary case
    if num_classes == 2:
        pred_flat = seg_pred.flatten()
        gt_flat = batch_gts.flatten()
        mask = (gt_flat != ignore_index)
        pred_valid = pred_flat[mask]
        gt_valid = gt_flat[mask]
        
        for p, g in zip(pred_valid, gt_valid):
            conf_mat[g, p] += 1

print("\nInference completed!")


Starting inference...
Number of classes: 2
Ignore index: 255


Testing:   0%|          | 0/4292 [00:00<?, ?it/s]

Testing: 100%|██████████| 4292/4292 [22:26<00:00,  3.19it/s]


Inference completed!


In [12]:
# Compute metrics
eval_metrics = evaluator.compute()

print("="*50)
print("Test Results:")
print("="*50)
print(f"mIoU:  {eval_metrics['mIoU']:.4f}")
print(f"aAcc:  {eval_metrics['aAcc']:.4f}")
print()


Test Results:
mIoU:  0.8273
aAcc:  0.9307



## 6. Binary Classification Metrics (for 2-class case)


In [13]:
if num_classes == 2:
    print("="*50)
    print("Binary Classification Metrics:")
    print("="*50)
    print("\nConfusion Matrix:")
    print(f"{'':>15} Pred_Negative  Pred_Positive")
    print(f"GT_Negative  {conf_mat[0][0]:>13}  {conf_mat[0][1]:>13}")
    print(f"GT_Positive  {conf_mat[1][0]:>13}  {conf_mat[1][1]:>13}")
    print()
    
    # Calculate metrics
    tn = conf_mat[0][0]
    fp = conf_mat[0][1]
    fn = conf_mat[1][0]
    tp = conf_mat[1][1]
    
    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)
    iou = tp / (tp + fp + fn + 1e-10)
    specificity = tn / (tn + fp + 1e-10)
    
    print("Metrics for Positive Class (Drivable Area):")
    print(f"Precision:    {precision:.4f}")
    print(f"Recall:       {recall:.4f}")
    print(f"F1-Score:     {f1:.4f}")
    print(f"IoU:          {iou:.4f}")
    print(f"Specificity:  {specificity:.4f}")
    print()
    
    # Overall accuracy
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-10)
    print(f"Overall Accuracy: {accuracy:.4f}")
else:
    print("Binary classification metrics are only available for 2-class problems.")


Binary Classification Metrics:

Confusion Matrix:
                Pred_Negative  Pred_Positive
GT_Negative     1159835334       27414065
GT_Positive       15179971      203973190

Metrics for Positive Class (Drivable Area):
Precision:    0.8815
Recall:       0.9307
F1-Score:     0.9055
IoU:          0.8273
Specificity:  0.9769

Overall Accuracy: 0.9697


## 7. Per-Class Results


In [14]:
# Get per-class results
per_class_results = evaluator.get_per_class_results()

print("="*50)
print("Per-Class Results:")
print("="*50)

class_names = ['Background', 'Drivable Area'] if num_classes == 2 else [f'Class_{i}' for i in range(num_classes)]

for i in range(num_classes):
    print(f"\n{class_names[i]}:")
    if 'per_class_iou' in per_class_results:
        print(f"  IoU:  {per_class_results['per_class_iou'][i]:.4f}")
    if 'per_class_acc' in per_class_results:
        print(f"  Acc:  {per_class_results['per_class_acc'][i]:.4f}")
    if 'per_class_dice' in per_class_results:
        print(f"  Dice: {per_class_results['per_class_dice'][i]:.4f}")


Per-Class Results:

Background:
  IoU:  0.9646
  Acc:  0.9769

Drivable Area:
  IoU:  0.8273
  Acc:  0.9307


## 8. Visualize Sample Predictions (Optional)


In [ ]:
# Uncomment to visualize sample predictions
# import matplotlib.pyplot as plt
# from mmcv.image import tensor2imgs

# # Get a sample batch
# sample_data = next(iter(test_loader))
# imgs = [img.to(device) for img in sample_data['img']]
# img_metas = [meta.data[0] for meta in sample_data['img_metas']]

# # Run inference
# with torch.no_grad():
#     seg_pred, _, _ = model.forward_test(imgs, img_metas, cfg.get('test_cfg'))
#     if seg_pred is not None:
#         seg_pred = seg_pred[0]

# # Load GT
# img_meta = img_metas[0][0]
# gt_path = os.path.join(test_dataset.ann_dir, img_meta['ann']['seg_map'])
# gt_seg = mmcv.imread(gt_path, flag='unchanged', backend='pillow')

# # Convert image
# img_tensor = imgs[0]
# img_vis = tensor2imgs(img_tensor, **img_meta['img_norm_cfg'])[0]

# # Plot
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# axes[0].imshow(img_vis)
# axes[0].set_title('Input Image')
# axes[0].axis('off')

# axes[1].imshow(gt_seg, cmap='gray')
# axes[1].set_title('Ground Truth')
# axes[1].axis('off')

# axes[2].imshow(seg_pred[0], cmap='gray')
# axes[2].set_title('Prediction')
# axes[2].axis('off')

# plt.tight_layout()
# plt.show()
